# Proyecto 2 - Modelado Predictivo de Mortalidad (Fases 0-5)

Este notebook implementa las primeras 5 fases del plan de prediccion y mantiene separado el Proyecto 1 (EDA/Clustering) del Proyecto 2 (Modelos).

Fases cubiertas aqui:
- Fase 0: Reutilizacion controlada
- Fase 1: Definir variable objetivo
- Fase 2: Antecedentes (plantilla academica)
- Fase 3: Preparacion de datos
- Fase 4: Train/Validation/Test split
- Fase 5: Seleccion de algoritmos


## Fase 0 - Reutilizacion controlada

Este notebook reutiliza logica de limpieza ya alineada con el Proyecto 1 (ejemplo: tratamiento de `Edadif == 999` y manejo de columnas entre anos).

Se mantiene separado de `main.ipynb` para:
- evitar notebooks demasiado largos,
- no afectar el flujo EDA/Clustering,
- facilitar revision independiente en GitHub y por el profesor.


In [1]:
from __future__ import annotations

from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
import pyreadstat

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
RANDOM_STATE = 42
YEARS = list(range(2013, 2023))
DATA_DIR = Path('data/defunciones')

# Opcional para prototipar rapido en equipos con menos RAM
USE_SAMPLE = False
SAMPLE_SIZE = 200_000

np.random.seed(RANDOM_STATE)

In [3]:
def normalize_text(value: str) -> str:
    if value is None:
        return ''
    value = unicodedata.normalize('NFKD', str(value))
    value = ''.join(ch for ch in value if not unicodedata.combining(ch))
    return value.lower().strip()


def resolve_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    norm_map = {normalize_text(col): col for col in df.columns}
    for candidate in candidates:
        key = normalize_text(candidate)
        if key in norm_map:
            return norm_map[key]
    return None


def normalize_cie10(code: object) -> str | None:
    if pd.isna(code):
        return None
    text = str(code).upper().strip()
    match = re.search(r'[A-Z][0-9]{2}', text)
    return match.group(0) if match else None


def map_causa_grupo(cie10: str | None) -> str | None:
    """Agrupa CIE-10 en macrogrupos consolidados para modelado predictivo."""
    if cie10 is None:
        return None

    letter = cie10[0]
    number = int(cie10[1:3]) if cie10[1:3].isdigit() else None

    # 1) Infecciosas
    if letter in {'A', 'B'}:
        return 'Infecciosas'

    # 2) Cronicas no transmisibles (neoplasias, sangre/inmunidad, endocrinas, circulatorias)
    if letter == 'C':
        return 'Cronicas_no_transmisibles'
    if letter == 'D' and number is not None and (0 <= number <= 89):
        return 'Cronicas_no_transmisibles'
    if letter in {'E', 'I'}:
        return 'Cronicas_no_transmisibles'

    # 3) Respiratorias
    if letter == 'J':
        return 'Respiratorias'

    # 4) Digestivo y genitourinario
    if letter in {'K', 'N'}:
        return 'Digestivo_genitourinario'

    # 5) Externas y trauma
    if letter in {'S', 'T', 'V', 'W', 'X', 'Y'}:
        return 'Externas_trauma'

    # 6) Materno infantil
    if letter in {'O', 'P', 'Q'}:
        return 'Materno_infantil'

    # 7) Otros capitulos
    if letter in {'F', 'G', 'H', 'L', 'M', 'R', 'U', 'Z'}:
        return 'Otros'

    return 'Otros'


def load_yearly_data(data_dir: Path, years: list[int]) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for year in years:
        path = data_dir / f'{year}.sav'
        if not path.exists():
            print(f'[WARN] No existe: {path}')
            continue
        df_year, _ = pyreadstat.read_sav(str(path))
        df_year['__year_file__'] = year
        frames.append(df_year)
        print(f'[OK] {year}: {len(df_year):,} registros')

    if not frames:
        raise FileNotFoundError('No se pudieron cargar archivos .sav en data/defunciones')

    return pd.concat(frames, ignore_index=True)


In [4]:
df = load_yearly_data(DATA_DIR, YEARS)
print(f'\nShape consolidado: {df.shape}')

if USE_SAMPLE and len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f'Se usa muestra para prototipado: {df.shape}')

[OK] 2013: 76,639 registros
[OK] 2014: 77,807 registros
[OK] 2015: 80,876 registros
[OK] 2016: 82,565 registros
[OK] 2017: 81,726 registros
[OK] 2018: 83,071 registros
[OK] 2019: 85,600 registros
[OK] 2020: 96,001 registros
[OK] 2021: 118,465 registros
[OK] 2022: 95,386 registros

Shape consolidado: (878136, 30)


## Fase 1 - Variable objetivo

**Variable respuesta seleccionada:** `grupo_causa_cie10` (derivada de `Caudef` CIE-10).

- Tipo: **cualitativa nominal** (clasificacion multiclase).
- Clases propuestas (consolidadas): `Infecciosas`, `Cronicas_no_transmisibles`, `Respiratorias`, `Digestivo_genitourinario`, `Externas_trauma`, `Materno_infantil`, `Otros`.
- Justificacion: conserva mas detalle clinico que una agrupacion de 3 clases y sigue siendo tratable con ML supervisado.



Fuente de referencia usada para rangos: `diccionario.md` (tabla CIE-10) y estructura de capitulos ICD-10 de OMS.

- Razon del enfoque consolidado: reduce desbalance extremo y facilita comparar modelos sin perder interpretabilidad epidemiologica.


In [5]:
caudef_col = resolve_column(df, ['Caudef'])
if caudef_col is None:
    raise KeyError('No se encontro la columna Caudef para construir la variable objetivo.')

df['cie10_norm'] = df[caudef_col].apply(normalize_cie10)
df['grupo_causa_cie10'] = df['cie10_norm'].apply(map_causa_grupo)

print('Distribucion inicial de la variable objetivo:')
display(df['grupo_causa_cie10'].value_counts(dropna=False).to_frame('conteo'))

df = df[df['grupo_causa_cie10'].notna()].copy()
print(f'\nRegistros tras remover objetivo nulo: {len(df):,}')

Distribucion inicial de la variable objetivo:


,conteo
grupo_causa_cie10,
Cronicas_no_transmisibles,328894
Otros,155802
Externas_trauma,115560
Digestivo_genitourinario,104241
Respiratorias,87441
Materno_infantil,44778
Infecciosas,41420



Registros tras remover objetivo nulo: 878,136


## Fase 2 - Antecedentes (plantilla academica)

Objetivo de esta seccion: construir minimo 2 paginas de antecedentes con referencias APA verificables de fuentes confiables (IEEE, EBSCOhost, ResearchGate, etc.).

Estructura sugerida para redactar:
1. Estado del arte en prediccion de mortalidad y causas de muerte.
2. Variables mas usadas y tecnicas de preprocesamiento reportadas.
3. Algoritmos mas frecuentes y metricas de comparacion.
4. Brecha que este proyecto cubre para Guatemala 2013-2022.

Checklist minimo:
- 8-12 articulos cientificos recientes (idealmente 2018+).
- Tabla comparativa de estudios.
- Referencias en formato APA 7 consistentes con citas en texto.


In [6]:
tabla_antecedentes = pd.DataFrame(
    columns=[
        'autor_anio',
        'fuente',
        'objetivo',
        'dataset',
        'algoritmos',
        'metrica_principal',
        'hallazgo_clave',
        'limitacion',
        'referencia_apa',
    ]
)

display(tabla_antecedentes)
print('Completar esta tabla con los articulos revisados.')

,autor_anio,fuente,objetivo,dataset,algoritmos,metrica_principal,hallazgo_clave,limitacion,referencia_apa


Completar esta tabla con los articulos revisados.


## Fase 3 - Preparacion de datos

En esta fase se construye una matriz de modelado sin fuga de informacion:
- no se usa `Caudef` como feature (origen directo de la etiqueta),
- se limpian sentinelas (`Edadif == 999`),
- se generan variables temporales simples para mejorar senal predictiva.


In [7]:
sexo_col = resolve_column(df, ['Sexo'])
edad_col = resolve_column(df, ['Edadif'])
depocu_col = resolve_column(df, ['Depocu', 'Depreg'])
mupocu_col = resolve_column(df, ['Mupocu', 'Mupreg'])
mes_col = resolve_column(df, ['Mesocu', 'Mesreg'])
dia_col = resolve_column(df, ['Diaocu'])
anio_col = resolve_column(df, ['Añoocu', 'Anoocu', 'Añoreg', 'Anoreg'])
ecivil_col = resolve_column(df, ['Ecidif'])
escolar_col = resolve_column(df, ['Escodif'])
ocup_col = resolve_column(df, ['Ciuodif'])

feature_map = {
    'sexo': sexo_col,
    'edad': edad_col,
    'departamento': depocu_col,
    'municipio': mupocu_col,
    'mes': mes_col,
    'dia': dia_col,
    'anio': anio_col,
    'estado_civil': ecivil_col,
    'escolaridad': escolar_col,
    'ocupacion': ocup_col,
}

missing_features = [k for k, v in feature_map.items() if v is None]
print('Columnas detectadas:')
display(pd.Series(feature_map, name='columna_en_dataset'))
if missing_features:
    print(f'[WARN] No detectadas: {missing_features}')

selected_pairs = [(k, v) for k, v in feature_map.items() if v is not None]
df_model = df[[v for _, v in selected_pairs]].copy()
df_model.columns = [k for k, _ in selected_pairs]

# Limpieza principal heredada de practicas del Proyecto 1
if 'edad' in df_model.columns:
    df_model['edad'] = pd.to_numeric(df_model['edad'], errors='coerce')
    df_model.loc[df_model['edad'] == 999, 'edad'] = np.nan

if 'dia' in df_model.columns:
    df_model['dia'] = pd.to_numeric(df_model['dia'], errors='coerce')

if 'mes' in df_model.columns:
    df_model['mes'] = pd.to_numeric(df_model['mes'], errors='coerce')

if 'anio' in df_model.columns:
    df_model['anio'] = pd.to_numeric(df_model['anio'], errors='coerce')

# Feature temporal simple
if {'dia', 'mes', 'anio'}.issubset(df_model.columns):
    fecha = pd.to_datetime(
        dict(year=df_model['anio'], month=df_model['mes'], day=df_model['dia']),
        errors='coerce',
    )
    df_model['es_fin_semana'] = fecha.dt.dayofweek.isin([5, 6]).astype('float')

y = df['grupo_causa_cie10'].copy()

print(f'Shape de modelado: X={df_model.shape}, y={y.shape}')

Columnas detectadas:


sexo               Sexo
edad             Edadif
departamento     Depocu
municipio        Mupocu
mes              Mesocu
dia              Diaocu
anio             Añoocu
estado_civil     Ecidif
escolaridad     Escodif
ocupacion       Ciuodif
Name: columna_en_dataset, dtype: str

Shape de modelado: X=(878136, 11), y=(878136,)


In [8]:
print('Balance de clases global:')
display(y.value_counts(normalize=True).mul(100).round(2).to_frame('%'))

print('Nulos por variable (top 10):')
display(df_model.isna().mean().sort_values(ascending=False).head(10).to_frame('%_nulos'))

Balance de clases global:


,%
grupo_causa_cie10,
Cronicas_no_transmisibles,37.45
Otros,17.74
Externas_trauma,13.16
Digestivo_genitourinario,11.87
Respiratorias,9.96
Materno_infantil,5.10
Infecciosas,4.72


Nulos por variable (top 10):


,%_nulos
anio,0.175879
edad,0.005893
sexo,0.000000
departamento,0.000000
municipio,0.000000
mes,0.000000
dia,0.000000
estado_civil,0.000000
escolaridad,0.000000
ocupacion,0.000000


## Fase 4 - Train/Validation/Test split

Estrategia aplicada:
- 70% entrenamiento
- 15% validacion
- 15% prueba

Como la variable objetivo es categorica, se usa `stratify=y` para mantener proporcion de clases entre particiones.


In [9]:
X_train, X_temp, y_train, y_temp = train_test_split(
    df_model,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

print('Shapes:')
print(f'  Train: {X_train.shape} | {y_train.shape}')
print(f'  Valid: {X_valid.shape} | {y_valid.shape}')
print(f'  Test : {X_test.shape} | {y_test.shape}')

def class_share(series: pd.Series) -> pd.Series:
    return series.value_counts(normalize=True).mul(100).round(2)

balance = pd.concat(
    [
        class_share(y_train).rename('train_%'),
        class_share(y_valid).rename('valid_%'),
        class_share(y_test).rename('test_%'),
    ],
    axis=1,
).fillna(0.0)

display(balance)

Shapes:
  Train: (614695, 11) | (614695,)
  Valid: (131720, 11) | (131720,)
  Test : (131721, 11) | (131721,)


,train_%,valid_%,test_%
grupo_causa_cie10,,,
Cronicas_no_transmisibles,37.45,37.45,37.45
Otros,17.74,17.74,17.74
Externas_trauma,13.16,13.16,13.16
Digestivo_genitourinario,11.87,11.87,11.87
Respiratorias,9.96,9.96,9.96
Materno_infantil,5.10,5.10,5.10
Infecciosas,4.72,4.72,4.72


## Fase 5 - Seleccion de algoritmos

Se seleccionan 3 algoritmos de clasificacion para la siguiente fase de entrenamiento/tuning:

1. **Regresion Logistica**: baseline interpretable y robusto para tabular.
2. **Random Forest**: captura no linealidad e interacciones sin escalar demasiado la data.
3. **Extra Trees**: variante de arboles con mayor aleatoriedad, util para comparar generalizacion.

La Fase 6 (no incluida aun) debe crear al menos 3 variaciones por cada algoritmo para elegir el mejor sin sobreajuste.


In [10]:
numeric_features = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
categorical_features = [c for c in X_train.columns if c not in numeric_features]

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=True)

numeric_pipe = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_pipe = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', ohe),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipe, numeric_features),
        ('cat', categorical_pipe, categorical_features),
    ]
)

algoritmos = {
    'logistic_regression': LogisticRegression(
        max_iter=1200,
        class_weight='balanced',
        random_state=RANDOM_STATE,
    ),
    'random_forest': RandomForestClassifier(
        n_estimators=300,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    'extra_trees': ExtraTreesClassifier(
        n_estimators=400,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

pipelines = {
    name: Pipeline(steps=[('preprocess', preprocessor), ('model', model)])
    for name, model in algoritmos.items()
}

print('Algoritmos seleccionados para Fase 6:')
for name in pipelines:
    print(f' - {name}')

Algoritmos seleccionados para Fase 6:
 - logistic_regression
 - random_forest
 - extra_trees
